# Bank Customer Churn Analysis

This notebook performs simple exploratory data analysis for a banking churn dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style="whitegrid")

data_path = Path("../data/BankChurn.csv")
df = pd.read_csv(data_path)
print(df.head())

In [ ]:
# Clean column names
df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]

# Clean values
df = df.copy()
df["gender"] = df["gender"].astype(str).str.strip().str.title()
df["geography"] = df["geography"].astype(str).str.strip().str.title()

# Create age groups
bins = [0, 30, 40, 50, 100]
labels = ["Under 30", "31-40", "41-50", "51+"]
df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels, include_lowest=True)

# Fill missing values with median or mode
for col in ["credit_score", "age", "balance", "estimated_salary"]:
    if df[col].isna().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

df = df.drop_duplicates()
print(df.isna().sum())

In [ ]:
# Summary statistics
print(df.describe(include="all"))

In [ ]:
# 1. Churn rate
churn_rate = df["churned"].mean()
print(f"Churn rate: {churn_rate:.2%}")

In [ ]:
# 2. Churn by geography
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
fig.suptitle("Bank Customer Churn Analysis", fontsize=16)

sns.countplot(data=df, x="geography", hue="churned", ax=axes[0, 0])
axes[0, 0].set_title("Churn by Geography")

# 3. Churn by gender
sns.countplot(data=df, x="gender", hue="churned", ax=axes[0, 1])
axes[0, 1].set_title("Churn by Gender")

# 4. Churn by age group
sns.countplot(data=df, x="age_group", hue="churned", ax=axes[0, 2])
axes[0, 2].set_title("Churn by Age Group")

# 5. Churn by products
sns.countplot(data=df, x="num_of_products", hue="churned", ax=axes[0, 3])
axes[0, 3].set_title("Churn by Products")

# 6. Active members
sns.countplot(data=df, x="is_active_member", hue="churned", ax=axes[1, 0])
axes[1, 0].set_title("Churn by Active Member Status")

# 7. Balance distribution
sns.histplot(data=df, x="balance", hue="churned", kde=True, ax=axes[1, 1])
axes[1, 1].set_title("Balance Distribution")

# 8. Salary distribution
sns.histplot(data=df, x="estimated_salary", hue="churned", kde=True, ax=axes[1, 2])
axes[1, 2].set_title("Salary Distribution")

# 9. Credit score distribution
sns.histplot(data=df, x="credit_score", hue="churned", kde=True, ax=axes[1, 3])
axes[1, 3].set_title("Credit Score Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# 10. Correlation heatmap
numeric_cols = ["credit_score", "age", "tenure", "balance", "num_of_products", "estimated_salary", "churned"]
corr = df[numeric_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()